In [0]:
from pyspark.sql import functions as F

bronze = spark.table("nyc_taxi.bronze.yellow_tripdata")

DISTANCE_CEILING = 100
TOLLS_CEILING     = 100

bronze = bronze.withColumn("_payment_type_safe", F.coalesce(F.col("payment_type"), F.lit(-1)))

tagged = bronze.withColumn("_reject_reason",
    F.when(F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull(),
           "missing_timestamp")
     .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"),
           "dropoff_before_pickup")
     .when(F.year(F.col("tpep_pickup_datetime")) != 2024,
           "implausible_pickup_year")
     .when(F.col("trip_distance") <= 0,
           "zero_or_negative_distance")
     .when(F.col("trip_distance") > DISTANCE_CEILING,
           "implausible_distance")
     .when(F.col("passenger_count").isNull(),
           "null_passenger_count")
     .when(F.col("passenger_count") == 0,
           "zero_passenger_count")
     .when(F.col("passenger_count") > 6,
           "implausible_passenger_count")
     .when(F.col("tolls_amount") > TOLLS_CEILING,
           "implausible_tolls")
     .when(F.col("fare_amount") < 0,
           "negative_fare")
     .when((F.col("fare_amount") < 3.00) & (~F.col("_payment_type_safe").isin(3, 4)),
           "fare_below_minimum")
     .when(F.col("total_amount") < 0,
           "negative_total")
     .when((F.col("total_amount") < 4.50) & (~F.col("_payment_type_safe").isin(3, 4)),
           "total_below_minimum")
     .otherwise(None)
).drop("_payment_type_safe")

clean      = tagged.filter(F.col("_reject_reason").isNull()).drop("_reject_reason")
quarantine = tagged.filter(F.col("_reject_reason").isNotNull())

clean = (clean.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
              .withColumn("trip_duration_min",
                  (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
              .withColumn("_silver_processed_at", F.current_timestamp()))

key_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID", "trip_distance", "total_amount"]
clean = clean.withColumn("trip_id",
    F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_cols]), 256))

print("clean:     ", clean.count())
print("quarantine:", quarantine.count())
quarantine.groupBy("_reject_reason").count().orderBy(F.desc("count")).show()

In [0]:
%sql
DROP TABLE IF EXISTS nyc_taxi.silver.trips_clean;
DROP TABLE IF EXISTS nyc_taxi.gold.fact_trips;

In [0]:
(clean.write.format("delta").partitionBy("pickup_date")
    .saveAsTable("nyc_taxi.silver.trips_clean"))

(quarantine.write.format("delta").mode("overwrite")
    .saveAsTable("nyc_taxi.silver.trips_quarantine"))

print("silver rows:", spark.table("nyc_taxi.silver.trips_clean").count())

In [0]:
silver = spark.table("nyc_taxi.silver.trips_clean")
zone_keys = [r.zone_key for r in spark.table("nyc_taxi.gold.dim_taxi_zone").select("zone_key").collect()]

fact = (silver
    .withColumn("date_key", F.col("pickup_date"))
    .withColumn("pickup_zone_key",
        F.when(F.col("PULocationID").isin(zone_keys), F.col("PULocationID")).otherwise(F.lit(-1)))
    .withColumn("dropoff_zone_key",
        F.when(F.col("DOLocationID").isin(zone_keys), F.col("DOLocationID")).otherwise(F.lit(-1)))
    .withColumn("rate_code_key",
        F.coalesce(F.col("RatecodeID").cast("int"), F.lit(-1)))
    .withColumn("payment_type_key",
        F.coalesce(F.col("payment_type").cast("int"), F.lit(-1)))
    .select(
        "trip_id", "date_key", "pickup_zone_key", "dropoff_zone_key",
        "rate_code_key", "payment_type_key",
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "passenger_count", "trip_distance", "trip_duration_min",
        "fare_amount", "tip_amount", "total_amount")
)

(fact.write.format("delta")
    .partitionBy("date_key")
    .saveAsTable("nyc_taxi.gold.fact_trips"))

print("fact_trips rows:", spark.table("nyc_taxi.gold.fact_trips").count())